In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
import librosa
import numpy as np
import pandas as pd
from torch import nn 
from torch.utils.data  import Dataset , DataLoader
from sklearn.preprocessing import LabelEncoder

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("USING : " , device)

In [ ]:
base = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
train_path = os.path.join(base , "genres_stems")
test = os.path.join(base , "mashups")

In [ ]:
genres = sorted(os.listdir(train_path))
X_train =[]
y_train = [] 
def load_and_merge_stems(song_path):
    stems = []
    for stem in ["drums.wav", "vocals.wav" ,"bass.wav" , "other.wav"]:
        path = os.path.join(song_path , stem)
        y , sr = librosa.load(path, sr=22050)
        stems.append(y)
    merged = np.sum(stems , axis =0)
    return merged
    
print("Loading training data...")

for genre in genres :
    genre_path = os.path.join(train_path , genre)
    songs = os.listdir(genre_path)

    for song in songs[:10]:
        song_path = os.path.join(genre_path , song)
        audio = load_and_merge_stems(song_path)
        audio = librosa.util.fix_length(audio , size = 22050*5)
        mel = librosa.feature.melspectrogram(y=audio , sr = 22050 , n_mels = 128)
        mel_db= librosa.power_to_db(mel)

        X_train.append(mel_db)
        y_train.append(genre)

print("loaded")

In [ ]:
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)

X_train = np.array(X_train)
X_train = X_train[: , None , : , :]

X_train = torch.tensor(X_train , dtype=torch.float32)
y_train = torch.tensor(y_train , dtype=torch.long)

train_dataset = torch.utils.data.TensorDataset(X_train , y_train)
train_loader = DataLoader(train_dataset , batch_size = 8 , shuffle=True)


In [ ]:
data = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv")
data.to_csv("submission.csv" , index = False)